# MathVision Peak Demand Prediction

Uses **Prophet** (Meta's time-series forecasting library) to predict session volume for the next 7 days, broken down by time slot (morning / afternoon / evening).

**Pipeline:**
1. Load 2 years of historical session records
2. Aggregate daily session counts per time slot
3. Fit a Prophet model per slot — captures weekly + yearly seasonality
4. Forecast the next 7 days of expected session volume
5. Normalise forecasts to a 0–1 demand score
6. Identify peak periods and generate ramp-up staffing recommendations

The same logic runs live in `api/services/demand_predictor.py` via `GET /demand/predict`.

## 1) Imports and setup

In [ ]:
import csv
import warnings
from datetime import date, datetime, timedelta
from math import ceil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from prophet import Prophet

warnings.filterwarnings('ignore')  # suppress Prophet/Stan verbosity

CSV_PATH      = '../data/raw/pairings_raw.csv'
STUDENTS_PATH = '../data/raw/students.csv'

TIME_SLOTS    = ['morning', 'afternoon', 'evening']
DAY_LABELS    = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
PEAK_THRESHOLD = 0.75
FORECAST_DAYS  = 7

SLOT_COLORS = {'morning': '#4A90D9', 'afternoon': '#EF9F27', 'evening': '#7B5EA7'}

print('Setup complete.')

## 2) Load and inspect session data

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=['session_date'])
df = df.dropna(subset=['session_date', 'duration_hours'])
df['time_slot'] = df['time_slot'].where(df['time_slot'].isin(TIME_SLOTS), other=None)

# Derive time_slot from pairing_id hash for any rows missing it
slot_map = {0: 'morning', 1: 'afternoon', 2: 'evening'}
mask = df['time_slot'].isna()
df.loc[mask, 'time_slot'] = df.loc[mask, 'pairing_id'].apply(lambda pid: slot_map[hash(pid) % 3])

print(f'Total sessions: {len(df):,}')
print(f'Date range:     {df["session_date"].min().date()} → {df["session_date"].max().date()}')
print(f'Time slot dist:\n{df["time_slot"].value_counts()}')
df.head()

## 3) Aggregate daily counts per time slot

Prophet needs a `ds` (date) and `y` (value) column. We count sessions per day per slot.

In [ ]:
def build_daily_series(df, slot):
    sub = df[df['time_slot'] == slot].copy()
    daily = sub.groupby('session_date').size().reset_index(name='y')
    daily = daily.rename(columns={'session_date': 'ds'})
    # Fill missing dates with 0 so Prophet sees a continuous series
    full_range = pd.date_range(daily['ds'].min(), daily['ds'].max(), freq='D')
    daily = daily.set_index('ds').reindex(full_range, fill_value=0).reset_index()
    daily.columns = ['ds', 'y']
    return daily

series = {slot: build_daily_series(df, slot) for slot in TIME_SLOTS}

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, slot in zip(axes, TIME_SLOTS):
    s = series[slot]
    ax.plot(s['ds'], s['y'].rolling(7).mean(), color=SLOT_COLORS[slot], linewidth=1.5, label='7-day avg')
    ax.fill_between(s['ds'], s['y'], alpha=0.15, color=SLOT_COLORS[slot])
    ax.set_ylabel('Sessions', fontsize=9)
    ax.set_title(f'{slot.capitalize()} slot', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.suptitle('Historical daily session volume by time slot', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4) Fit Prophet models

One model per time slot. Prophet automatically learns:
- **Weekly seasonality** — which days of the week are busiest
- **Yearly seasonality** — exam-season spikes in March and October
- **Trend** — whether overall demand is growing or flat

In [ ]:
models   = {}
forecasts = {}

for slot in TIME_SLOTS:
    s = series[slot]
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.1,
    )
    m.fit(s)
    future   = m.make_future_dataframe(periods=FORECAST_DAYS)
    forecast = m.predict(future)
    models[slot]    = m
    forecasts[slot] = forecast
    print(f'{slot:12s} — fitted on {len(s)} days')

print('\nAll models fitted.')

## 5) Visualise forecasts with uncertainty bands

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)

for ax, slot in zip(axes, TIME_SLOTS):
    fc    = forecasts[slot]
    hist  = series[slot]
    color = SLOT_COLORS[slot]

    # Historical
    ax.plot(hist['ds'], hist['y'], color='#aaaaaa', linewidth=0.8, alpha=0.6, label='Actual')

    # Fitted + forecast
    ax.plot(fc['ds'], fc['yhat'].clip(lower=0), color=color, linewidth=1.5, label='Forecast')
    ax.fill_between(fc['ds'], fc['yhat_lower'].clip(lower=0), fc['yhat_upper'].clip(lower=0),
                    alpha=0.2, color=color, label='Uncertainty')

    # Highlight forecast window
    cutoff = hist['ds'].max()
    ax.axvline(cutoff, color='#E24B4A', linestyle='--', linewidth=1, label='Forecast start')

    ax.set_title(f'{slot.capitalize()} — 7-day forecast', fontsize=10, fontweight='bold')
    ax.set_ylabel('Sessions / day', fontsize=9)
    ax.legend(fontsize=8, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.suptitle('Prophet demand forecast by time slot', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6) Weekly seasonality components

Shows which days of the week each slot is naturally busiest — useful for validating the model makes intuitive sense.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, slot in zip(axes, TIME_SLOTS):
    fc    = forecasts[slot]
    color = SLOT_COLORS[slot]

    # Extract the last 7 forecast days as a proxy for weekly pattern
    week = fc.tail(7).copy()
    week['dow'] = week['ds'].dt.day_name()
    week = week.set_index('dow').reindex(
        ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    )

    bars = ax.bar(range(7), week['yhat'].clip(lower=0), color=color, alpha=0.8)
    ax.set_xticks(range(7))
    ax.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], fontsize=9)
    ax.set_title(f'{slot.capitalize()}', fontsize=10, fontweight='bold')
    ax.set_ylabel('Forecast sessions', fontsize=9)

    # Annotate bars
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}',
                    ha='center', va='bottom', fontsize=7)

plt.suptitle('Forecast session volume — next 7 days by slot', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7) Build demand matrix and identify peaks

In [ ]:
# Collect all yhat values across all slots to normalise
all_yhats = []
for slot in TIME_SLOTS:
    fc = forecasts[slot].tail(FORECAST_DAYS)
    all_yhats.extend(fc['yhat'].clip(lower=0).tolist())

max_yhat = max(all_yhats) if all_yhats else 1.0

matrix_rows = []
for slot in TIME_SLOTS:
    fc   = forecasts[slot].tail(FORECAST_DAYS).reset_index(drop=True)
    row  = []
    for i, r in fc.iterrows():
        yhat  = max(float(r['yhat']), 0.0)
        score = yhat / max_yhat if max_yhat > 0 else 0.0
        row.append({
            'date':       r['ds'].date(),
            'day_label':  r['ds'].strftime('%A'),
            'time_slot':  slot,
            'forecast':   round(yhat, 1),
            'lower':      round(max(float(r['yhat_lower']), 0.0), 1),
            'upper':      round(max(float(r['yhat_upper']), 0.0), 1),
            'demand_score': round(score, 3),
            'is_peak':    score >= PEAK_THRESHOLD,
        })
    matrix_rows.append(row)

# Display as a readable table
flat = [cell for row in matrix_rows for cell in row]
matrix_df = pd.DataFrame(flat)
print(f'Peak cells: {matrix_df["is_peak"].sum()} / {len(matrix_df)}')
display(matrix_df[['day_label','time_slot','forecast','lower','upper','demand_score','is_peak']])

## 8) Demand heatmap — next 7 days

In [ ]:
score_grid = pd.DataFrame(
    [[cell['demand_score'] for cell in row] for row in matrix_rows],
    index=[s.capitalize() for s in TIME_SLOTS],
    columns=[matrix_rows[0][i]['day_label'][:3] + '\n' + str(matrix_rows[0][i]['date']) for i in range(FORECAST_DAYS)]
)

fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(score_grid.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(FORECAST_DAYS))
ax.set_xticklabels(score_grid.columns, fontsize=9)
ax.set_yticks(range(3))
ax.set_yticklabels(score_grid.index, fontsize=9)

for i, row in enumerate(matrix_rows):
    for j, cell in enumerate(row):
        label = f"~{cell['forecast']:.0f}"
        if cell['is_peak']:
            label += ' ⚠'
        color = 'white' if cell['demand_score'] > 0.6 else 'black'
        ax.text(j, i, label, ha='center', va='center', fontsize=8, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, label='Demand Score (0–1)', shrink=0.8)
ax.set_title(f'7-Day Demand Forecast Heatmap (peak threshold = {PEAK_THRESHOLD})', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 9) Ramp-up recommendations

In [ ]:
recs = []
for row in matrix_rows:
    for cell in row:
        if cell['is_peak']:
            additional = max(1, ceil(
                (cell['demand_score'] - PEAK_THRESHOLD) / (1.0 - PEAK_THRESHOLD) * 5
            ))
            recs.append({
                'Day':              cell['day_label'],
                'Slot':             cell['time_slot'].capitalize(),
                'Forecast sessions':cell['forecast'],
                'Range':            f"{cell['lower']:.0f}–{cell['upper']:.0f}",
                'Demand score':     cell['demand_score'],
                'Extra tutors':     additional,
            })

recs.sort(key=lambda r: r['Demand score'], reverse=True)
recs_df = pd.DataFrame(recs)

if len(recs_df) > 0:
    print(f'{len(recs_df)} peak periods identified:\n')
    display(recs_df)
else:
    print('No peak periods at this threshold.')

## 10) Recommendations chart

In [ ]:
if len(recs) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    labels  = [f"{r['Day'][:3]} {r['Slot'][:3]}" for r in recs]
    scores  = [r['Demand score'] for r in recs]
    tutors  = [r['Extra tutors'] for r in recs]
    colors  = ['#E24B4A' if s >= 0.9 else '#EF9F27' for s in scores]

    ax = axes[0]
    ax.barh(labels, scores, color=colors)
    ax.axvline(x=PEAK_THRESHOLD, color='#999', linestyle='--', linewidth=1, label=f'Threshold ({PEAK_THRESHOLD})')
    ax.set_xlabel('Demand Score')
    ax.set_title('Peak Period Demand Scores')
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    ax2 = axes[1]
    ax2.barh(labels, tutors, color='#41b6c4')
    ax2.set_xlabel('Extra Tutors Recommended')
    ax2.set_title('Ramp-Up Recommendations')
    ax2.invert_yaxis()
    ax2.set_xticks(range(0, max(tutors) + 2))

    plt.tight_layout()
    plt.show()
else:
    print('No peaks to visualise.')

## 11) Branch-level filtered forecast

In [ ]:
students_df = pd.read_csv(STUDENTS_PATH)
branch_map  = dict(zip(students_df['student_id'], students_df['branch']))
branches    = sorted(students_df['branch'].unique())
print(f'Branches: {branches}')

for branch in branches:
    branch_ids = {sid for sid, b in branch_map.items() if b == branch}
    sub = df[df['student_id'].isin(branch_ids)]
    print(f'\n── {branch} ({len(sub)} sessions) ──')

    for slot in TIME_SLOTS:
        slot_sub = sub[sub['time_slot'] == slot]
        daily = slot_sub.groupby('session_date').size()
        if len(daily) < 14:
            print(f'  {slot}: insufficient data ({len(daily)} days)')
            continue

        daily_df = daily.reset_index()
        daily_df.columns = ['ds', 'y']
        full_range = pd.date_range(daily_df['ds'].min(), daily_df['ds'].max(), freq='D')
        daily_df = daily_df.set_index('ds').reindex(full_range, fill_value=0).reset_index()
        daily_df.columns = ['ds', 'y']

        m = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                    daily_seasonality=False, seasonality_mode='multiplicative')
        m.fit(daily_df)
        future = m.make_future_dataframe(periods=FORECAST_DAYS)
        fc = m.predict(future).tail(FORECAST_DAYS)
        avg_forecast = fc['yhat'].clip(lower=0).mean()
        print(f'  {slot}: avg {avg_forecast:.1f} sessions/day forecast')

## 12) Model validation — cross-validation on last 4 weeks

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

print('Running cross-validation on afternoon slot (representative)...')
m_cv = models['afternoon']

# Holdout: last 28 days, evaluated every 7 days, trained on at least 180 days
df_cv = cross_validation(m_cv, initial='180 days', period='7 days', horizon='28 days')
metrics = performance_metrics(df_cv)

print('\nPerformance metrics (afternoon slot):')
display(metrics[['horizon','mae','rmse','mape']].round(3))